In [4]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# Load dataset
housing = fetch_california_housing(as_frame=True)
data = housing.frame

# Separate features and target
X = data.drop(columns=["MedHouseVal"])
y = data["MedHouseVal"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (16512, 8)
Testing features: (4128, 8)
Training target: (16512,)
Testing target: (4128,)


In [6]:

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

print("Current directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("src folder exists:", (PROJECT_ROOT / "src").exists())
print(
    "Feature engineering file exists:",
    (PROJECT_ROOT / "src" / "feature_engineering.py").exists()
)


Current directory: d:\ML_Projects\House_Price_Prediction\notebooks
Project root: d:\ML_Projects\House_Price_Prediction
src folder exists: True
Feature engineering file exists: True


In [7]:

from src.feature_engineering import add_engineered_features
from sklearn.preprocessing import FunctionTransformer

feature_engineering = FunctionTransformer(
    add_engineered_features,
    validate=False
)

X_train_fe = feature_engineering.transform(X_train)

print("Original features:", X_train.shape[1])
print("Features after engineering:", X_train_fe.shape[1])



Original features: 8
Features after engineering: 9


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    )
}

print("Models prepared:")
for name in models:
    print(name)

Models prepared:
Linear Regression
Random Forest
Gradient Boosting


In [10]:
model_pipelines = {
    "Linear Regression": Pipeline([
        ("feature_engineering", feature_engineering),
        ("scaler", StandardScaler()),
        ("model", models["Linear Regression"])
    ]),

    "Random Forest": Pipeline([
        ("feature_engineering", feature_engineering),
        ("model", models["Random Forest"])
    ]),

    "Gradient Boosting": Pipeline([
        ("feature_engineering", feature_engineering),
        ("model", models["Gradient Boosting"])
    ])
}

for name, pipeline in model_pipelines.items():
    print(f"\n{name}:")
    print(pipeline)


Linear Regression:
Pipeline(steps=[('feature_engineering',
                 FunctionTransformer(func=<function add_engineered_features at 0x000001AEBFCD3740>)),
                ('scaler', StandardScaler()), ('model', LinearRegression())])

Random Forest:
Pipeline(steps=[('feature_engineering',
                 FunctionTransformer(func=<function add_engineered_features at 0x000001AEBFCD3740>)),
                ('model', RandomForestRegressor(n_jobs=-1, random_state=42))])

Gradient Boosting:
Pipeline(steps=[('feature_engineering',
                 FunctionTransformer(func=<function add_engineered_features at 0x000001AEBFCD3740>)),
                ('model', GradientBoostingRegressor(random_state=42))])


In [11]:
for name, pipeline in model_pipelines.items():
    pipeline.fit(X_train, y_train)
    print(f"{name} trained successfully!")

Linear Regression trained successfully!
Random Forest trained successfully!
Gradient Boosting trained successfully!


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []

for name, pipeline in model_pipelines.items():
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

results_df = pd.DataFrame(results)

print(results_df.round(4))

               Model     MAE    RMSE  R2 Score
0  Linear Regression  0.5261  0.7292    0.5942
1      Random Forest  0.3278  0.5054    0.8051
2  Gradient Boosting  0.3675  0.5378    0.7793


In [13]:
results_df = results_df.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

print("Model ranking by RMSE:")
print(results_df.round(4))

best_model_name = results_df.loc[0, "Model"]
best_model = model_pipelines[best_model_name]

print("\nBest model:", best_model_name)

Model ranking by RMSE:
               Model     MAE    RMSE  R2 Score
0      Random Forest  0.3278  0.5054    0.8051
1  Gradient Boosting  0.3675  0.5378    0.7793
2  Linear Regression  0.5261  0.7292    0.5942

Best model: Random Forest


In [14]:
import joblib
from pathlib import Path

# Access the existing models folder
models_dir = Path("../models")

# Save the best trained pipeline
model_path = models_dir / "best_house_price_model.pkl"

joblib.dump(best_model, model_path)

print("Best model:", best_model_name)
print("Model saved at:", model_path.resolve())
print("Model saved successfully!")

Best model: Random Forest
Model saved at: D:\ML_Projects\House_Price_Prediction\models\best_house_price_model.pkl
Model saved successfully!


In [15]:

import joblib
from pathlib import Path

# Locate the project's models folder
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(exist_ok=True)

# Save the best trained pipeline
model_path = models_dir / "best_house_price_model.pkl"

joblib.dump(best_model, model_path)

print("Best model:", best_model_name)
print("Model saved at:", model_path)

Best model: Random Forest
Model saved at: d:\ML_Projects\House_Price_Prediction\models\best_house_price_model.pkl
